In [639]:
import pandas as pd
import openpyxl
import os
import glob, re

In [640]:
def read_excel(files, sheet):
    dfs = []
    for file in files:
        data = pd.read_excel(file, sheet_name=sheet, index_col=0)
        name = os.path.splitext(os.path.basename(file))[0]
        data['model'] = name
        match = re.search(r'lr[\d.]+_(\d+)_', name)
        if match:
            data['epochs'] = int(match.group(1))
        match_model = re.search(r'model_.*?_e_(.*?)_lr', name)
        if match_model:
            data['id'] = match_model.group(1)
        dfs.append(data)
    df = pd.concat(dfs,ignore_index=True)
    return df

In [641]:
def arrange_data(df, error):
    df = df.pivot(index='model', columns='error', values=error)
    return df

In [642]:
def min_vals(df, error):
    df1 = df.apply(lambda x: pd.Series({'model':x.idxmin(), error:x.min()})).T
    return df1

In [643]:
path = "model_Fe_Si_B_260311/**/test_res/*_test.xlsx"

In [644]:
files = glob.glob(path, recursive=True)

In [645]:
df = read_excel(files, 'errors')
df = df[['id', 'epochs']+[c for c in df.columns if c not in ['id', 'epochs']]]
df

,id,epochs,error,rmse,mae,r2,model
0,scmace,100,test_energy,210.180375,124.774518,0.760373,model_rnd_e_scmace_lr0.0001_100_10_test
1,scmace,100,train_energy,212.670122,104.387671,0.779457,model_rnd_e_scmace_lr0.0001_100_10_test
2,scmace,100,test_force,0.473279,0.266766,0.971340,model_rnd_e_scmace_lr0.0001_100_10_test
3,scmace,100,train_force,0.350139,0.264250,0.984159,model_rnd_e_scmace_lr0.0001_100_10_test
4,matpes,80,test_energy,43.019987,26.794243,0.989961,model_rnd_e_matpes_lr0.0001_80_10_test
5,matpes,80,train_energy,100.808691,27.309964,0.950446,model_rnd_e_matpes_lr0.0001_80_10_test
6,matpes,80,test_force,0.159534,0.120127,0.996744,model_rnd_e_matpes_lr0.0001_80_10_test
7,matpes,80,train_force,0.139589,0.107203,0.997482,model_rnd_e_matpes_lr0.0001_80_10_test
8,matpes,100,test_energy,41.652997,24.019415,0.990589,model_rnd_e_matpes_lr0.0001_100_10_test
9,matpes,100,train_energy,100.094452,23.863738,0.951146,model_rnd_e_matpes_lr0.0001_100_10_test


In [646]:
#df.loc[df['model']=='model_rnd_e_matpes_lr0.0001_40_10_test']

In [647]:
rmse = arrange_data(df, 'rmse')
mae = arrange_data(df, 'mae')
rmse

error,test_energy,test_force,train_energy,train_force
model,,,,
model_rnd_e_matpes_lr0.0001_100_10_test,41.652997,0.159079,100.094452,0.136716
model_rnd_e_matpes_lr0.0001_40_10_test,49.126072,0.162873,103.295626,0.152998
model_rnd_e_matpes_lr0.0001_60_10_test,46.366162,0.160293,102.324352,0.146102
model_rnd_e_matpes_lr0.0001_80_10_test,43.019987,0.159534,100.808691,0.139589
model_rnd_e_scmace_lr0.0001_100_10_test,210.180375,0.473279,212.670122,0.350139
model_rnd_e_scmace_lr0.0001_40_10_test,231.558780,0.636152,252.479919,0.484112
model_rnd_e_scmace_lr0.0001_60_10_test,239.880400,0.571505,248.010446,0.421373
model_rnd_e_scmace_lr0.0001_80_10_test,230.282754,0.520437,232.658891,0.385412


In [648]:
min_vals(rmse, 'rsme')

,model,rsme
error,,
test_energy,model_rnd_e_matpes_lr0.0001_100_10_test,41.652997
test_force,model_rnd_e_matpes_lr0.0001_100_10_test,0.159079
train_energy,model_rnd_e_matpes_lr0.0001_100_10_test,100.094452
train_force,model_rnd_e_matpes_lr0.0001_100_10_test,0.136716


In [649]:
min_vals(mae, 'mae')

,model,mae
error,,
test_energy,model_rnd_e_matpes_lr0.0001_100_10_test,24.019415
test_force,model_rnd_e_matpes_lr0.0001_100_10_test,0.119892
train_energy,model_rnd_e_matpes_lr0.0001_100_10_test,23.863738
train_force,model_rnd_e_matpes_lr0.0001_100_10_test,0.105124


In [658]:
scmace = df.query('id == "scmace"')
matpes = df.query('id == "matpes"')
#matpes

In [661]:
scmace.query('id == "scmace" & error == "test_energy"').sort_values('mae', ascending=True)

,id,epochs,error,rmse,mae,r2,model
0,scmace,100,test_energy,210.180375,124.774518,0.760373,model_rnd_e_scmace_lr0.0001_100_10_test
24,scmace,80,test_energy,230.282754,135.255193,0.712344,model_rnd_e_scmace_lr0.0001_80_10_test
28,scmace,40,test_energy,231.558780,140.661072,0.709147,model_rnd_e_scmace_lr0.0001_40_10_test
20,scmace,60,test_energy,239.880400,145.884953,0.687866,model_rnd_e_scmace_lr0.0001_60_10_test


In [663]:
matpes.query('id == "matpes" & error == "test_energy"').sort_values('mae', ascending=True)

,id,epochs,error,rmse,mae,r2,model
8,matpes,100,test_energy,41.652997,24.019415,0.990589,model_rnd_e_matpes_lr0.0001_100_10_test
4,matpes,80,test_energy,43.019987,26.794243,0.989961,model_rnd_e_matpes_lr0.0001_80_10_test
12,matpes,40,test_energy,49.126072,28.338427,0.986909,model_rnd_e_matpes_lr0.0001_40_10_test
16,matpes,60,test_energy,46.366162,30.922178,0.988339,model_rnd_e_matpes_lr0.0001_60_10_test


In [667]:
min_vals(matpes, 'mae')

,model,mae
id,4,matpes
epochs,12,40
error,4,test_energy
rmse,11.0,0.136716
mae,11.0,0.105124
r2,13.0,0.947971
model,8,model_rnd_e_matpes_lr0.0001_100_10_test


In [668]:
min_vals(scmace, 'mae')

,model,mae
id,0,scmace
epochs,28,40
error,0,test_energy
rmse,3.0,0.350139
mae,3.0,0.26425
r2,20.0,0.687866
model,0,model_rnd_e_scmace_lr0.0001_100_10_test


Reading the config errors

In [650]:
df_config =read_excel(files, 'config_errors')
df_config = df_config[['id', 'epochs', 'config', 'n_configs', 'mae', 'rmse', 'error', 'model']]
df_config

,id,epochs,config,n_configs,mae,rmse,error,model
0,scmace,100,Fe12B3Si3,48,392.463504,393.680934,test_energy,model_rnd_e_scmace_lr0.0001_100_10_test
1,scmace,100,Fe12Si4,12,134.133466,136.318493,test_energy,model_rnd_e_scmace_lr0.0001_100_10_test
2,scmace,100,Fe16,11,66.959629,72.981237,test_energy,model_rnd_e_scmace_lr0.0001_100_10_test
3,scmace,100,Fe34B10Si10,133,12.602748,15.474716,test_energy,model_rnd_e_scmace_lr0.0001_100_10_test
4,scmace,100,Fe8B4,8,448.953319,448.967178,test_energy,model_rnd_e_scmace_lr0.0001_100_10_test
...,...,...,...,...,...,...,...,...
75,scmace,40,Fe12B3Si3,132,326.479043,329.236748,train_energy,model_rnd_e_scmace_lr0.0001_40_10_test
76,scmace,40,Fe12Si4,39,156.856406,159.559893,train_energy,model_rnd_e_scmace_lr0.0001_40_10_test
77,scmace,40,Fe16,40,376.785670,382.605320,train_energy,model_rnd_e_scmace_lr0.0001_40_10_test
78,scmace,40,Fe34B10Si10,590,16.111629,20.371132,train_energy,model_rnd_e_scmace_lr0.0001_40_10_test


In [651]:
df_config.query('id == "scmace" & error == "test_energy"').sort_values('mae', ascending=True)

,id,epochs,config,n_configs,mae,rmse,error,model
3,scmace,100,Fe34B10Si10,133,12.602748,15.474716,test_energy,model_rnd_e_scmace_lr0.0001_100_10_test
63,scmace,80,Fe34B10Si10,133,14.210503,17.948904,test_energy,model_rnd_e_scmace_lr0.0001_80_10_test
53,scmace,60,Fe34B10Si10,133,17.149657,21.801379,test_energy,model_rnd_e_scmace_lr0.0001_60_10_test
73,scmace,40,Fe34B10Si10,133,17.445929,21.840337,test_energy,model_rnd_e_scmace_lr0.0001_40_10_test
62,scmace,80,Fe16,11,62.649272,82.973017,test_energy,model_rnd_e_scmace_lr0.0001_80_10_test
2,scmace,100,Fe16,11,66.959629,72.981237,test_energy,model_rnd_e_scmace_lr0.0001_100_10_test
61,scmace,80,Fe12Si4,12,130.369046,133.245172,test_energy,model_rnd_e_scmace_lr0.0001_80_10_test
1,scmace,100,Fe12Si4,12,134.133466,136.318493,test_energy,model_rnd_e_scmace_lr0.0001_100_10_test
51,scmace,60,Fe12Si4,12,150.652201,153.529932,test_energy,model_rnd_e_scmace_lr0.0001_60_10_test
71,scmace,40,Fe12Si4,12,153.629668,156.097101,test_energy,model_rnd_e_scmace_lr0.0001_40_10_test


In [652]:
df_config.query('id == "scmace" & error == "train_energy"').sort_values('mae', ascending=True)

,id,epochs,config,n_configs,mae,rmse,error,model
8,scmace,100,Fe34B10Si10,590,12.430298,15.470571,train_energy,model_rnd_e_scmace_lr0.0001_100_10_test
68,scmace,80,Fe34B10Si10,590,13.863354,17.120244,train_energy,model_rnd_e_scmace_lr0.0001_80_10_test
78,scmace,40,Fe34B10Si10,590,16.111629,20.371132,train_energy,model_rnd_e_scmace_lr0.0001_40_10_test
58,scmace,60,Fe34B10Si10,590,17.171249,20.989510,train_energy,model_rnd_e_scmace_lr0.0001_60_10_test
7,scmace,100,Fe16,40,56.930606,66.693207,train_energy,model_rnd_e_scmace_lr0.0001_100_10_test
67,scmace,80,Fe16,40,71.874366,88.522012,train_energy,model_rnd_e_scmace_lr0.0001_80_10_test
66,scmace,80,Fe12Si4,39,133.545638,136.543100,train_energy,model_rnd_e_scmace_lr0.0001_80_10_test
6,scmace,100,Fe12Si4,39,137.138739,139.183731,train_energy,model_rnd_e_scmace_lr0.0001_100_10_test
56,scmace,60,Fe12Si4,39,154.173300,157.262401,train_energy,model_rnd_e_scmace_lr0.0001_60_10_test
76,scmace,40,Fe12Si4,39,156.856406,159.559893,train_energy,model_rnd_e_scmace_lr0.0001_40_10_test


In [653]:
df_config.query('id == "matpes" & error == "test_energy"').sort_values('mae', ascending=True)

,id,epochs,config,n_configs,mae,rmse,error,model
32,matpes,40,NaN,11,3.363690,4.621235,test_energy,model_rnd_e_matpes_lr0.0001_40_10_test
23,matpes,100,Fe34B10Si10,133,3.642926,4.661568,test_energy,model_rnd_e_matpes_lr0.0001_100_10_test
42,matpes,60,NaN,11,3.799697,4.854654,test_energy,model_rnd_e_matpes_lr0.0001_60_10_test
41,matpes,60,NaN,12,3.927227,4.348577,test_energy,model_rnd_e_matpes_lr0.0001_60_10_test
33,matpes,40,NaN,133,4.299808,5.523249,test_energy,model_rnd_e_matpes_lr0.0001_40_10_test
11,matpes,80,NaN,12,4.808195,5.172613,test_energy,model_rnd_e_matpes_lr0.0001_80_10_test
12,matpes,80,NaN,11,6.094816,6.651858,test_energy,model_rnd_e_matpes_lr0.0001_80_10_test
22,matpes,100,Fe16,11,6.880569,7.642087,test_energy,model_rnd_e_matpes_lr0.0001_100_10_test
13,matpes,80,NaN,133,8.338329,9.337885,test_energy,model_rnd_e_matpes_lr0.0001_80_10_test
21,matpes,100,Fe12Si4,12,9.839741,10.007741,test_energy,model_rnd_e_matpes_lr0.0001_100_10_test


In [655]:
df_config.query('id == "matpes" & error == "train_energy"').sort_values('mae', ascending=True)

,id,epochs,config,n_configs,mae,rmse,error,model
28,matpes,100,Fe34B10Si10,590,3.656830,4.759407,train_energy,model_rnd_e_matpes_lr0.0001_100_10_test
46,matpes,60,NaN,39,4.228887,4.652940,train_energy,model_rnd_e_matpes_lr0.0001_60_10_test
38,matpes,40,NaN,590,4.554277,5.775843,train_energy,model_rnd_e_matpes_lr0.0001_40_10_test
37,matpes,40,NaN,40,4.875505,6.571228,train_energy,model_rnd_e_matpes_lr0.0001_40_10_test
16,matpes,80,NaN,39,4.905662,5.244371,train_energy,model_rnd_e_matpes_lr0.0001_80_10_test
47,matpes,60,NaN,40,5.088528,6.387306,train_energy,model_rnd_e_matpes_lr0.0001_60_10_test
17,matpes,80,NaN,40,7.774841,8.987460,train_energy,model_rnd_e_matpes_lr0.0001_80_10_test
18,matpes,80,NaN,590,8.296824,9.358866,train_energy,model_rnd_e_matpes_lr0.0001_80_10_test
27,matpes,100,Fe16,40,8.827048,10.186386,train_energy,model_rnd_e_matpes_lr0.0001_100_10_test
26,matpes,100,Fe12Si4,39,9.987663,10.192224,train_energy,model_rnd_e_matpes_lr0.0001_100_10_test
